# RS-Flow-VQA: Continuous Soft-Prefix Flow Matching & FreeFlow Distillation

This notebook demonstrates the complete end-to-end pipeline for **RS-Flow-VQA**:
1. **Data Preparation**: Loading & tokenizing RSICD dataset captions.
2. **Feature & Embedding Caching**: Extracting Scale-MAE 1024-dim image features and Qwen token lookup table.
3. **Whitening Normalization**: Computing per-channel mean and standard deviation for prompt embedding whitening.
4. **Conditional Flow Matching (CFM) Teacher**: Training CFM teacher $v_\phi$ with Minibatch OT coupling.
5. **Target-Free FreeFlow Distillation**: Distilling into 1-step student $f_\theta$ using discrete prediction ($N=8$) and auxiliary correction $g_\psi$.
6. **Evaluation & Zero-Shot VQA Transfer**: Metric calculation on RSICD captions and zero-shot VQA on RSVQA-LR.

In [ ]:
# Install package in editable mode if running in Colab / Jupyter
!pip install -e . --quiet

In [ ]:
import torch
from rs_flow_vqa.config import load_config
from rs_flow_vqa.training.train_teacher import train_teacher_pipeline
from rs_flow_vqa.training.distill_freeflow import distill_freeflow_pipeline
from rs_flow_vqa.evaluation.eval_caption import evaluate_caption_pipeline
from rs_flow_vqa.evaluation.eval_rsvqa import evaluate_rsvqa_pipeline

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())

## 1. Run Pipeline in Smoke Mode
We load the smoke configuration and execute all pipeline stages.

In [ ]:
# Load smoke configuration
cfg = load_config(smoke=True, device_override='cpu')
print('Loaded Experiment:', cfg.experiment_name)

In [ ]:
# Step 1: Feature caching
from rs_flow_vqa.cli import cache_features_cmd
import argparse

args = argparse.Namespace(config=None, smoke=True, device='cpu', seed=42, output_dir='./outputs/smoke_run')
cache_features_cmd(args)

In [ ]:
# Step 2: Train CFM Teacher
teacher_ckpt = train_teacher_pipeline(cfg)

In [ ]:
# Step 3: Target-Free FreeFlow Distillation
student_ckpt = distill_freeflow_pipeline(cfg)

In [ ]:
# Step 4: Evaluate Caption Quality & Fidelity
cap_metrics = evaluate_caption_pipeline(cfg)

In [ ]:
# Step 5: Evaluate Zero-Shot VQA Transfer on RSVQA-LR
rsvqa_metrics = evaluate_rsvqa_pipeline(cfg)

## 2. Summary Metrics Table
Print summary metrics table comparing Baselines, 16-NFE Teacher, and 1-Step FreeFlow Student.

In [ ]:
import pandas as pd

data = [
    {'Model': 'Text-Only Qwen Baseline', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['text_only_baseline']['overall']*100:.2f}%", 'Bridge Latency (ms)': '0.0 ms', 'NFEs': 0},
    {'Model': 'CFM Teacher (16-NFE)', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['teacher_16nfe']['overall']*100:.2f}%", 'Bridge Latency (ms)': f"{cap_metrics['teacher_16nfe_latency_ms']:.2f} ms", 'NFEs': 16},
    {'Model': 'FreeFlow Student (1-Step)', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['student_1step']['overall']*100:.2f}%", 'Bridge Latency (ms)': f"{cap_metrics['student_1step_latency_ms']:.2f} ms", 'NFEs': 1},
]
df = pd.DataFrame(data)
display(df)